In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [2]:
def wait_time_t2(i,j,k,L1, L2, R1, c1, R2, c2, P, mu):
    n_star = math.floor(mu * ((R1-P) / c1))
    T = 0
    p = L1 / mu
    n = i + k
    K = n_star + 1
    t1 = (1/mu)*(1/(1-p)**2)*( n*(1-p) - (1-p**n)*p**(K-n+1) )

    m = 1
    t2 = (1/mu)*(1/(1-p)**2)*( m*(1-p) - (1-p**m)*p**(K-m+1) )

    T = t1 + t2*j

    return T

In [3]:
def util2(i,j,k,L1, L2, R1, c1, R2, c2, P, mu):
    util = R2 - P - c2*wait_time_t2(i,j,k,L1, L2, R1, c1, R2, c2, P, mu)
    return util

In [4]:
def will_enter_t2_state_dependent(i,j,k,L1, L2, R1, c1, R2, c2, P, mu):
    U = util2(i,j,k,L1, L2, R1, c1, R2, c2, P, mu)

    if U >=0:
        return 1
    else:
        return 0

In [5]:
def find_space_prev(L1, L2, R1, c1, R2, c2, P, mu, n_star):
    state_space = [(i, 0, 1) for i in range(n_star+1)]
    x = 0
    y = len(state_space)
    
    while x < y:
        if will_enter_t2_state_dependent(state_space[x][0], state_space[x][1], state_space[x][2],L1, L2, R1, c1, R2, c2, P, mu) == 1:
            state_space.append( ( state_space[x][0], state_space[x][1] + 1, state_space[x][2] ) )
    
        y = len(state_space)
        x += 1
    
    for s in state_space:
        a,b,c = s
        for i in range(n_star+1):
            if (i,b,c) not in state_space:
                state_space.append((i,b,c))
    
    
    state_space = [(0,0,0)] + state_space
    
    return state_space

In [6]:
def find_decision_set(L1, L2, R1, c1, R2, c2, P, mu):
    n_star = math.floor(mu * ((R1-P) / c1))
    t1_list = []
    for n in range(0, n_star+1):
        t1_list.append(n)
    
    index = {}
    final_list = []
    for n in t1_list:
        state_t2_prev = find_space_prev(L1, L2, R1, c1, R2, c2, P, mu, n)
        index[n] = list(range(n + 2)) 

        from itertools import product
        max_list = {}
        sub_list = {}
        
        for i, j, k in state_t2_prev:
            if (i,k) == (0,0):
                max_list[(i, k)] = 1
            else:
                s_total = i + j + k
                if ((i, k) not in max_list) or (s_total > max_list[(i, k)]):
                    max_list[(i, k)] = s_total
        
        
        for i in sorted(max_list):
            start = 1 if i == (0, 0) else 0
            sub_list[i] = [(i, s) for s in range(start, max_list[i] + 1)]

        
        
        i_sorted = sorted(sub_list)
        all_combos = [list(tup) for tup in product(*(sub_list[i] for i in i_sorted))]
        
        
        for f in all_combos:
                final_list.append((n, f))

    return final_list

       


In [7]:
x = find_decision_set(1, 2, 1, 1, 2, 2, 0, 3.1)

In [8]:
def will_enter_type2(i,j,k, decision):
    matrix = decision
    target = i, k
    result = next((x for x in matrix if x[0] == target), None)
    m_star = result[1]

    if i+j+k < m_star:
        return 1
    else:
        return 0


In [9]:
import numpy as np
from collections import deque

def utility(L1, L2, R1, c1, R2, c2, P, mu, decision):
    n_star, type2_matrix = decision

    # -----------------------------
    # TYPE 1
    # -----------------------------
    type1_probs = {}
    π0 = (mu - L1) / (mu + L2 - (L1 + L2) * ((L1 / mu) ** (n_star + 1)))
    type1_probs[0] = π0
    for n in range(1, n_star + 2):
        π_n = π0 * ((L1 / mu) ** (n - 1)) * ((L1 + L2) / mu)
        type1_probs[n] = π_n
    result1 = type1_probs

    h1 = 1 - result1[(n_star + 1)]

    U1 = 0.0
    W_1 = 0
    for s in list(result1.keys())[:-1]:
        W1_now = (s / mu)
        W_1  += result1[s]*W1_now
        U1 +=  result1[s] * (R1 - c1 * W1_now)

    # -----------------------------
    # TYPE 2: state space
    # -----------------------------
    state_space = []
    for m in type2_matrix:
        i, k, t_max = m[0][0], m[0][1], m[1]
        j_ye_kalan = t_max - i - k
        if j_ye_kalan < 0:
            j_ye_kalan = 0
        for j in range(0, j_ye_kalan + 1):
            state_space.append((i, j, k))

    if (0, 0, 1) not in state_space:
        state_space.append((0, 0, 1))
    if (0, 0, 0) not in state_space:
        state_space.append((0, 0, 0))

    states = state_space
    total_states = len(states)
    index_map = {st: idx for idx, st in enumerate(states)}

    transition_matrix = np.zeros((total_states, total_states))

    for idx, (i, j, k) in enumerate(states):
        cs = idx

        # k == 0 -> (0,0,1)
        if k == 0:
            ns = index_map.get((0, 0, 1))
            if ns is not None:
                if P > R1 and P <= R2:
                    transition_matrix[cs, ns] = L2
                elif P > R2 and P <= R1:
                    transition_matrix[cs, ns] = L1
                else:
                    transition_matrix[cs, ns] = L1 + L2

        # k != 0 and i==0 and j==0
        if k != 0 and i == 0 and j == 0:
            ns = index_map.get((0, 0, 0))
            if ns is not None:
                transition_matrix[cs, ns] = mu

            if k + i <= n_star and P <= R1:
                ns = index_map.get((1, 0, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L1

            if will_enter_type2(i, j, k, type2_matrix) == 1 and P <= R2:
                ns = index_map.get((0, 1, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L2

        # j==0 and i!=0
        if j == 0 and i != 0:
            ns = index_map.get((i - 1, j, 1))
            if ns is not None:
                transition_matrix[cs, ns] = mu

            if k + i <= n_star and P <= R1:
                ns = index_map.get((i + 1, j, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L1

            if will_enter_type2(i, j, k, type2_matrix) == 1 and P <= R2:
                ns = index_map.get((i, j + 1, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L2

        # j!=0 and i==0
        if j != 0 and i == 0:
            ns = index_map.get((i, j - 1, 1))
            if ns is not None:
                transition_matrix[cs, ns] = mu

            if k + i <= n_star and P <= R1:
                ns = index_map.get((i + 1, j, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L1

            if will_enter_type2(i, j, k, type2_matrix) == 1 and P <= R2:
                ns = index_map.get((i, j + 1, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L2

        # j!=0 and i!=0
        if j != 0 and i != 0:
            ns = index_map.get((i - 1, j, 1))
            if ns is not None:
                transition_matrix[cs, ns] = mu

            if k + i <= n_star and P <= R1:
                ns = index_map.get((i + 1, j, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L1

            if will_enter_type2(i, j, k, type2_matrix) == 1 and P <= R2:
                ns = index_map.get((i, j + 1, 1))
                if ns is not None:
                    transition_matrix[cs, ns] = L2

    # -----------------------------
    # Q matrix (CTMC generator)
    # -----------------------------
    Q = transition_matrix.copy()
    n = Q.shape[0]
    for r in range(n):
        Q[r, r] = -np.sum(transition_matrix[r, :])

    start_state = (0, 0, 1) if (0, 0, 1) in index_map else (0, 0, 0)
    s0 = index_map[start_state]

    eps = 1e-12
    neighbors = [list(np.where(Q[i, :] > eps)[0]) for i in range(n)]

    visited = np.zeros(n, dtype=bool)
    q = deque([s0])
    visited[s0] = True
    while q:
        u = q.popleft()
        for v in neighbors[u]:
            if not visited[v]:
                visited[v] = True
                q.append(v)

    reach_idx = np.where(visited)[0]
    if reach_idx.size == 0:
        reach_idx = np.arange(n)

    Qr = Q[np.ix_(reach_idx, reach_idx)]
    states_r = [states[i] for i in reach_idx]

    # -----------------------------
    # Stationary on reachable chain
    # -----------------------------
    dim = Qr.shape[0]
    A = np.vstack([Qr.T, np.ones(dim)])
    bb = np.zeros(dim + 1)
    bb[-1] = 1.0
    pi, *_ = np.linalg.lstsq(A, bb, rcond=None)

    
    pi = np.maximum(pi, 0.0)
    ssum = float(pi.sum())
    if ssum == 0.0:
        raise np.linalg.LinAlgError("Stationary computation failed: pi sums to 0 after clipping.")
    pi = pi / ssum

    result_dict = {st: float(p) for st, p in zip(states_r, pi)}

    # -----------------------------
    # Type-2 joining probability h2
    # -----------------------------
    type2_joining_rate = 0.0
    for st, prob in result_dict.items():
        i, j, k = st
        if will_enter_type2(i, j, k, type2_matrix) == 1:
            type2_joining_rate += prob

    if P > R2:
        type2_joining_rate = 0.0

    h2 = type2_joining_rate

    # -----------------------------
    # Utility of type 2
    # -----------------------------
    U2 = 0.0
    W_2 = 0
    for st, prob in result_dict.items():
        i, j, k = st
        if will_enter_type2(i, j, k, type2_matrix) == 1:
            W2_now = wait_time_t2(i, j, k, L1, L2, R1, c1, R2, c2, P, mu)
            W_2  += prob*W2_now
            U2 += prob * (R2 - c2 * W2_now)

    return U1, U2, h1, h2, W_1, W_2


    

In [10]:
utility(1, 2, 1, 1, 2, 2, 0, 3.1, decision = (0, [((0, 0), 0), ((0, 1), 1)]))

(0.5081967213114755, 0.0, 0.5081967213114753, 0.0, 0.0, 0)

In [11]:
def find_max_util(L1, L2, R1, c1, R2, c2, P, mu):
    dec_set = find_decision_set(L1, L2, R1, c1, R2, c2, P, mu)

    best_value = float("-inf")
    best_decision = None
    
    for dec in dec_set:
        U1, U2, h1, h2, W_1, W_2 = utility(L1, L2, R1, c1, R2, c2, P, mu, dec)
        U11 = L1*h1*U1
        U22 = L2*h2*U2
        total = U11 + U22

        if total > best_value:
            best_value = total
            best_decision = dec
            best_h = h1, h2
            best_U = U1, U2
            best_W = W_1, W_2


    return best_decision, best_value, best_h, U1, U2, best_W

In [12]:
find_max_util(1, 1, 5, 7, 10, 3, 0, 3)

((1, [((0, 0), 1), ((0, 1), 7), ((1, 1), 7)]),
 11.245379198320204,
 (0.8823529411764706, 0.9939568186216241),
 3.490566037735849,
 7.718337439468425,
 (0.1176470588235294, 0.6037191643859586))